# Solving a PDE

In this example, we solve the Cahn–Hilliard equation on a two-dimensional periodic domain and animate the result. In Colab, use the default CPU or select an available GPU under **Runtime → Change runtime type**, then select **Runtime → Run all**. The setup cell installs `mosaix-pde` if needed. The solve cell prints the device holding the solution.

In [ ]:
import sys
from importlib.metadata import version
from importlib.util import find_spec

if find_spec('mosaix_pde') is None:
    # Keep Colab's JAX version paired with its accelerator libraries.
    jax_requirement = "jax"
    if "google.colab" in sys.modules and find_spec("jax") is not None:
        jax_requirement = f"jax=={version('jax')}"
    %pip install {jax_requirement} git+https://github.com/acoh64/mosaix-pde.git

In [1]:
import jax.numpy as jnp
from jax import random
import matplotlib.pyplot as plt
from IPython.display import HTML
import matplotlib.animation as animation

from mosaix_pde.numerics.domains import Domain
from mosaix_pde.numerics.equations.cahn_hilliard import CahnHilliard2DPeriodic
from mosaix_pde.numerics.solvers import SemiImplicitFourierSpectral
from mosaix_pde.pde_model import PDEModel

First, we create the domain that we will solve the PDE on.

In [2]:
Nx, Ny = 128, 128
Lx = 0.01 * Nx
Ly = 0.01 * Ny
domain = Domain((Nx, Ny), ((-Lx / 2, Lx / 2), (-Ly / 2, Ly / 2)), "dimensionless")

We combine the domain and the Cahn–Hilliard equation in a PDE model and use a semi-implicit Fourier spectral method for time stepping.

In [3]:
model = PDEModel(CahnHilliard2DPeriodic, domain, SemiImplicitFourierSpectral)

We save the solution at 200 evenly spaced times between the start and end of the simulation.

In [4]:
t_start = 0.0
t_final = 0.2
ts_save = jnp.linspace(t_start, t_final, 200)

We also need to specify the parameters in the PDE and solver

In [5]:
pde_parameters = {
    "kappa": 0.002,
    "mu": lambda c: jnp.log(c / (1.0 - c)) + 3.0 * (1.0 - 2.0 * c),
    "D": lambda c: (1.0 - c) * c,
    "derivs": "fd"
}

solver_parameters = {
    "A": 0.5,
}

Finally, let's create an initial condition for the simulation.

In [6]:
key = random.PRNGKey(0)
u0 = 0.5 * jnp.ones((Nx, Ny)) + 0.01 * random.normal(key, (Nx, Ny))

In [7]:
solution = model.solve(
    pde_parameters,
    u0,
    ts_save,
    solver_parameters,
)
device = solution.device
print(f"Solution device: {device.platform} ({device.device_kind})")

In [8]:
fig, ax = plt.subplots(figsize=(4,4))

ims = []
for i in range(0, len(solution), 2):
    im = ax.imshow(solution[i], animated=True, cmap='RdBu', 
                   vmin=0.0, vmax=1.0,
                   extent=[domain.box[0][0], domain.box[0][1], 
                          domain.box[1][0], domain.box[1][1]])
    ims.append([im])

ani = animation.ArtistAnimation(fig, ims, interval=100, blit=True)

plt.title('Cahn-Hilliard Evolution')
plt.xlabel('x')
plt.ylabel('y')

plt.close()

HTML(ani.to_jshtml())
